# Project name - FMCG Sales Drivers Analysis

## Causal and Statistical Analysis of Sales Drivers in FMCG Retail: A Multivariate Study

# 1. Problem Definition (Scientific Method)

## Research Question
    What affects sales?
    Which operational, environmental, and store-level factors significantly drive FMCG beverage sales?

Why it matters (business + analytics)

## Hypotheses
    H1: Store visits positively affect sales volume
    H2: Weather conditions (temperature, precipitation) significantly influence demand
    H3: Store execution (coolers, visibility, equipment) increases sales performance
    H4: Effects differ across store segments and regions

# 2. DATA ARCHITECTURE
## CORE DATA (Business Layer)
### 1. SALES (target dataset)
    date
    customer
    revenue_bgn
    cases
## OPERATIONS LAYER
### 2. VISITS
    Customer
    date
    visits   
### 3. COOLER / EQUIPMENT
    customer
    equipment count
    branding
    status
## MASTER STORE DATA
### 4. LIST DATASET
    customer
    region
    city
    channel
    segment
## EXTERNAL ENVIRONMENT
### 5. WEATHER
    date
    temperature
    precipitation
    humidity
    wind
    holiday
    weekend

    
internal sales system

weather dataset (external source)

holidays dataset

DATA LOADING + DATA QUALITY CHECK

In [1]:
import pandas as pd

# =========================
# LOAD DATASETS
# =========================
sales = pd.read_csv("data/sales_dataset_clean.csv.gz")
visits = pd.read_csv("data/visits_dataset.csv")
cooler = pd.read_csv("data/cooler_dataset.csv")
store = pd.read_csv("data/list_dataset.csv")
weather = pd.read_csv("data/weather_dataset_bg.csv")

# =========================
# CLEAN COLUMN NAMES
# =========================
for df in [sales, visits, cooler, store, weather]:
    df.columns = df.columns.str.strip()

# =========================
# RENAME COLUMNS (IMPORTANT FIX)
# =========================

# VISITS
visits = visits.rename(columns={
    "Calendar day": "date",
    "Customer": "customer",
    "Executed\nVisits All": "visits"
})

# STORE (IMPORTANT FIX: Customer -> customer)
store = store.rename(columns={
    "Customer": "customer"
})

# COOLER
cooler = cooler.rename(columns={
    "Customer": "customer",
    "Number of \nEquipments": "equipment_count"
})

# =========================
# FIX DATE TYPES
# =========================
sales["date"] = pd.to_datetime(sales["date"])
visits["date"] = pd.to_datetime(visits["date"], dayfirst=True)
weather["date"] = pd.to_datetime(weather["date"])

# =========================
# SAFETY: REMOVE DUPLICATE CUSTOMER COLS
# =========================
cooler = cooler.drop_duplicates(subset=["customer"])

DATA INTEGRATION

In [2]:
# =========================
# STEP 1: SALES + VISITS
# =========================
df = sales.merge(visits, on=["customer", "date"], how="left")

# =========================
# STEP 2: ADD STORE INFO
# =========================
df = df.merge(store, on="customer", how="left")

# =========================
# STEP 3: ADD COOLER DATA
# =========================
df = df.merge(cooler, on="customer", how="left")

# =========================
# STEP 4: ADD WEATHER
# =========================
df = df.merge(weather, on="date", how="left")

# =========================
# RESULT CHECK
# =========================
print(df.head())
print("Shape:", df.shape)
print(df.columns.tolist())

        date  customer revenue_bgn cases  visits Customer Name  Region  \
0 2025-01-01    103335        13,9     5     NaN  МАГАЗИН ЕООД  Добрич   
1 2025-01-01    103335        13,9     5     NaN  МАГАЗИН ЕООД  Добрич   
2 2025-01-01    103335        13,9     5     NaN  МАГАЗИН ЕООД  Добрич   
3 2025-01-01    103335        13,9     5     NaN  МАГАЗИН ЕООД  Добрич   
4 2025-01-01    103335        13,9     5     NaN  МАГАЗИН ЕООД  Добрич   

       City Trade Channel Segment  ... equipment_count Number of doors  \
0  ЛЮЛЯКОВО    Local Shop  Bronze  ...             1.0               1   
1  ЛЮЛЯКОВО    Local Shop  Bronze  ...             1.0               1   
2  ЛЮЛЯКОВО    Local Shop  Bronze  ...             1.0               1   
3  ЛЮЛЯКОВО    Local Shop  Bronze  ...             1.0               1   
4  ЛЮЛЯКОВО    Local Shop  Bronze  ...             1.0               1   

  temperature_celsius precipitation_mm humidity_percent  wind_speed_kmh  \
0                 2.9              

 FEATURE ENGINEERING

In [3]:
# =========================
# FEATURE ENGINEERING
# =========================

df["sales_per_case"] = df["revenue_bgn"] / (df["cases"] + 1)

df["sales_per_visit"] = df["revenue_bgn"] / (df["visits"] + 1)

df["high_visit"] = df["visits"] > df["visits"].median()

df["temperature_bucket"] = pd.cut(
    df["temperature_celsius"],
    bins=[-10, 10, 25, 40],
    labels=["cold", "mild", "hot"]
)

df["is_active_store"] = df["status"].notna()

TypeError: can only concatenate str (not "int") to str

EDA

In [ ]:
import matplotlib.pyplot as plt

# SALES DISTRIBUTION
df["revenue_bgn"].hist()
plt.title("Sales Distribution")
plt.show()

# VISITS vs SALES
plt.scatter(df["visits"], df["revenue_bgn"])
plt.title("Visits vs Sales")
plt.show()

# TEMPERATURE vs SALES
plt.scatter(df["temperature_celsius"], df["revenue_bgn"])
plt.title("Temperature vs Sales")
plt.show()

3. Data Structure Overview

4. Loading Data

5. Project Plan 

Data cleaning

EDA

Hypothesis testing

Regression modeling

Conclusions